# This Notebook estimates the model

## Settings

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize

import DynamicTimeAllocationModel

# c++ settings
do_compile = True
threads = 64

import os
os.environ.pop('NoDefaultCurrentDirectoryInExePath', None)

do_reinstall_nlopt = False # If problems with NLOPT during re-compilation of c++ files, then try re-installing NLOPT by swithcing this to True and delete the folder "nlopt-2.4.2-dll64" in the cppfuncs folder before running this notebook.
if do_reinstall_nlopt:
    from EconModel import cpptools
    cpptools.setup_nlopt(folder='cppfuncs/', do_print=False,download=False,unzip=True)

In [2]:
# setup model
settings = { 
       # technical settings
       'threads':threads,
       'do_multistart': False,
       'do_egm': True,
       'interp_method': 'linear',
       'interp_inverse': True,
       'precompute_intratemporal': True,
       'centered_gradient': True,
       'bargaining': 'limited',
}


model = DynamicTimeAllocationModel.HouseholdModelClass(par=settings) 
model.link_to_cpp(force_compile=do_compile)

## Empirical Moments to Match

In [3]:
# all moments listed here will be used in estimation. Comment out those you do not want to use.
datamoms = dict()

# wages
datamoms['wage_level_w_25_34'] = 40.1
datamoms['wage_level_w_35_44'] = 49.3
datamoms['wage_level_m_25_34'] = 50.3
datamoms['wage_level_m_35_44'] = 67.8

# employment rates
datamoms['employment_rate_w_35_44'] = 64.0
datamoms['employment_rate_m_35_44'] = 88.0
datamoms['work_hours_w'] = 4.41*365 / 52.0  # daily hours to weekly hours
datamoms['work_hours_m'] = 5.7*365 / 52.0  # daily hours to weekly hours

# # consumption
datamoms['consumption'] = 42.716
datamoms['consumption_90_10_ratio'] = 3.33 * 1.0954

# # marriage and divorce rates
datamoms['marriage_rate_35_44'] = 69.0

# Mazzocco moments
datamoms['home_prod_w'] = (2.23+0.75+1.47+0.08) * 365/ 52
datamoms['home_prod_m'] = (1.6+0.54+0.88+0.1) * 365/ 52


# weights
weights = dict()
for mom in ('consumption_90_10_ratio',):
    weights[mom] = 10.0
    

## Parameters to estimate

In [4]:
# parameters to estimate
estpars = {
    # Wages
    'mu': {'guess':2.3678,'lower':0.1,'upper':3.00}, 
    'mu_mult': {'guess':1.1126,'lower':1.0,'upper':3.0},
    'gamma': {'guess':0.1237,'lower':0.001,'upper':0.50},
    'gamma_mult': {'guess':1.7611,'lower':1.0,'upper':3.0},
    'sigma_mu': {'guess':0.5613,'lower':0.001,'upper':1.0},
    
    # Disutility from work
    'eta': {'guess':0.9033,'lower':0.1,'upper':5.0},
    'eta_mult': {'guess':0.8877,'lower':0.3,'upper':3.0},
    'phi': {'guess':4.4732,'lower':0.1,'upper':5.0},
    'phi_mult': {'guess':1.0855,'lower':0.3,'upper':3.0},
    
    # Home production
    'alpha': {'guess':0.9608,'lower':0.1,'upper':1.9},
    'pi': {'guess':0.6144,'lower':0.1,'upper':0.9},
    'lambda_': {'guess':5.7527,'lower':0.1,'upper':30.0},
    
    # # Match quality
    'sigma_love': {'guess':3.7895,'lower':0.01,'upper':20.5},
}

## setup initial guess 

In [5]:
# check bounds
bounds_ok = True
for key in estpars.keys():
    if estpars[key]['guess']<estpars[key]['lower']:
        print(key,' lower',estpars[key]['guess'])
        bounds_ok = False
    
    if estpars[key]['guess']>estpars[key]['upper']:
        print(key,' upper',estpars[key]['guess'])
        bounds_ok = False

if not bounds_ok:
    stop

In [6]:
# check initial guess
theta_init = np.array([estpars[key]['guess'] for key in estpars.keys()])
obj_init = model.obj_func(theta_init, estpars, datamoms, weights, do_print=True)

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6279, data: 40.1000
  wage_level_w_35_44       : sim: 51.8705, data: 49.3000
  wage_level_m_25_34       : sim: 50.0504, data: 50.3000
  wage_level_m_35_44       : sim: 67.0223, data: 67.8000
  employment_rate_w_35_44  : sim: 63.8213, data: 64.0000
  employment_rate_m_35_44  : sim: 88.4377, data: 88.0000
  work_hours_w             : sim: 27.9590, data: 30.9548
  work_hours_m             : sim: 36.5939, data

## Estimate model

In [7]:
# Estimate model using nelder-mead algorithm
do_print = True
res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',
               options={'xatol': 1e-3, 'fatol': 1e-3, 'disp': True, 'maxiter':500, 'maxfev':500})


Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6279, data: 40.1000
  wage_level_w_35_44       : sim: 51.8705, data: 49.3000
  wage_level_m_25_34       : sim: 50.0504, data: 50.3000
  wage_level_m_35_44       : sim: 67.0223, data: 67.8000
  employment_rate_w_35_44  : sim: 63.8213, data: 64.0000
  employment_rate_m_35_44  : sim: 88.4377, data: 88.0000
  work_hours_w             : sim: 27.9590, data: 30.9548
  work_hours_m             : sim: 36.5939, data

Parameters:
  mu             : 2.4862 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.6943, data: 40.1000
  wage_level_w_35_44       : sim: 58.6766, data: 49.3000
  wage_level_m_25_34       : sim: 53.4895, data: 50.3000
  wage_level_m_35_44       : sim: 74.3345, data: 67.8000
  employment_rate_w_35_44  : sim: 62.1352, data: 64.0000
  employment_rate_m_35_44  : sim: 93.4340, data: 88.0000
  work_hours_w             : sim: 27.7690, data: 30.9548
  work_hours_m             : sim: 38.1326, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1682 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.3461, data: 40.1000
  wage_level_w_35_44       : sim: 53.7030, data: 49.3000
  wage_level_m_25_34       : sim: 53.2301, data: 50.3000
  wage_level_m_35_44       : sim: 73.4071, data: 67.8000
  employment_rate_w_35_44  : sim: 52.2316, data: 64.0000
  employment_rate_m_35_44  : sim: 95.0160, data: 88.0000
  work_hours_w             : sim: 25.1169, data: 30.9548
  work_hours_m             : sim: 38.5867, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1299 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9490, data: 40.1000
  wage_level_w_35_44       : sim: 52.7430, data: 49.3000
  wage_level_m_25_34       : sim: 49.2422, data: 50.3000
  wage_level_m_35_44       : sim: 67.6254, data: 67.8000
  employment_rate_w_35_44  : sim: 62.8751, data: 64.0000
  employment_rate_m_35_44  : sim: 91.8478, data: 88.0000
  work_hours_w             : sim: 27.8488, data: 30.9548
  work_hours_m             : sim: 37.6525, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.8492 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.3906, data: 40.1000
  wage_level_w_35_44       : sim: 52.3256, data: 49.3000
  wage_level_m_25_34       : sim: 48.7549, data: 50.3000
  wage_level_m_35_44       : sim: 67.3055, data: 67.8000
  employment_rate_w_35_44  : sim: 61.5520, data: 64.0000
  employment_rate_m_35_44  : sim: 92.7053, data: 88.0000
  work_hours_w             : sim: 27.4101, data: 30.9548
  work_hours_m             : sim: 37.9182, data

...

Parameters:
  mu             : 2.3842 (init: 2.3678)
  mu_mult        : 1.1231 (init: 1.1126)
  gamma          : 0.1116 (init: 0.1237)
  gamma_mult     : 1.8350 (init: 1.7611)
  sigma_mu       : 0.5504 (init: 0.5613)
  eta            : 0.9233 (init: 0.9033)
  eta_mult       : 0.8623 (init: 0.8877)
  phi            : 4.3231 (init: 4.4732)
  phi_mult       : 1.0899 (init: 1.0855)
  alpha          : 0.9287 (init: 0.9608)
  pi             : 0.6311 (init: 0.6144)
  lambda_        : 6.1546 (init: 5.7527)
  sigma_love     : 3.9113 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9428, data: 40.1000
  wage_level_w_35_44       : sim: 50.7835, data: 49.3000
  wage_level_m_25_34       : sim: 51.1082, data: 50.3000
  wage_level_m_35_44       : sim: 67.1074, data: 67.8000
  employment_rate_w_35_44  : sim: 63.5876, data: 64.0000
  employment_rate_m_35_44  : sim: 88.7509, data: 88.0000
  work_hours_w             : sim: 27.8867, data: 30.9548
  work_hours_m             : sim: 36.6124, data

Parameters:
  mu             : 2.3850 (init: 2.3678)
  mu_mult        : 1.1235 (init: 1.1126)
  gamma          : 0.1110 (init: 0.1237)
  gamma_mult     : 1.8396 (init: 1.7611)
  sigma_mu       : 0.5498 (init: 0.5613)
  eta            : 0.9246 (init: 0.9033)
  eta_mult       : 0.8624 (init: 0.8877)
  phi            : 4.3366 (init: 4.4732)
  phi_mult       : 1.0923 (init: 1.0855)
  alpha          : 0.9284 (init: 0.9608)
  pi             : 0.6309 (init: 0.6144)
  lambda_        : 6.1645 (init: 5.7527)
  sigma_love     : 3.9044 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9771, data: 40.1000
  wage_level_w_35_44       : sim: 50.7403, data: 49.3000
  wage_level_m_25_34       : sim: 51.1685, data: 50.3000
  wage_level_m_35_44       : sim: 67.1297, data: 67.8000
  employment_rate_w_35_44  : sim: 63.5585, data: 64.0000
  employment_rate_m_35_44  : sim: 88.7217, data: 88.0000
  work_hours_w             : sim: 27.8595, data: 30.9548
  work_hours_m             : sim: 36.5935, data

Parameters:
  mu             : 2.3828 (init: 2.3678)
  mu_mult        : 1.1230 (init: 1.1126)
  gamma          : 0.1118 (init: 0.1237)
  gamma_mult     : 1.8350 (init: 1.7611)
  sigma_mu       : 0.5516 (init: 0.5613)
  eta            : 0.9225 (init: 0.9033)
  eta_mult       : 0.8644 (init: 0.8877)
  phi            : 4.3193 (init: 4.4732)
  phi_mult       : 1.0879 (init: 1.0855)
  alpha          : 0.9293 (init: 0.9608)
  pi             : 0.6310 (init: 0.6144)
  lambda_        : 6.1532 (init: 5.7527)
  sigma_love     : 3.9296 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9284, data: 40.1000
  wage_level_w_35_44       : sim: 50.7922, data: 49.3000
  wage_level_m_25_34       : sim: 50.9942, data: 50.3000
  wage_level_m_35_44       : sim: 66.9932, data: 67.8000
  employment_rate_w_35_44  : sim: 63.5818, data: 64.0000
  employment_rate_m_35_44  : sim: 88.9270, data: 88.0000
  work_hours_w             : sim: 27.9032, data: 30.9548
  work_hours_m             : sim: 36.6765, data

Parameters:
  mu             : 2.3823 (init: 2.3678)
  mu_mult        : 1.1231 (init: 1.1126)
  gamma          : 0.1114 (init: 0.1237)
  gamma_mult     : 1.8390 (init: 1.7611)
  sigma_mu       : 0.5522 (init: 0.5613)
  eta            : 0.9229 (init: 0.9033)
  eta_mult       : 0.8663 (init: 0.8877)
  phi            : 4.3273 (init: 4.4732)
  phi_mult       : 1.0881 (init: 1.0855)
  alpha          : 0.9296 (init: 0.9608)
  pi             : 0.6309 (init: 0.6144)
  lambda_        : 6.1605 (init: 5.7527)
  sigma_love     : 3.9407 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9463, data: 40.1000
  wage_level_w_35_44       : sim: 50.7514, data: 49.3000
  wage_level_m_25_34       : sim: 50.9397, data: 50.3000
  wage_level_m_35_44       : sim: 66.8910, data: 67.8000
  employment_rate_w_35_44  : sim: 63.5611, data: 64.0000
  employment_rate_m_35_44  : sim: 89.1102, data: 88.0000
  work_hours_w             : sim: 27.9008, data: 30.9548
  work_hours_m             : sim: 36.7279, data

C:\Users\zbk883\AppData\Local\Temp\13\ipykernel_15872\1613049469.py:3: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',


In [8]:
# final solution: objective, parameters and simulated vs. data moments at the optimum
obj_final = model.obj_func(res.x, estpars, datamoms, weights, do_print=True)

Parameters:
  mu             : 2.3828 (init: 2.3678)
  mu_mult        : 1.1230 (init: 1.1126)
  gamma          : 0.1118 (init: 0.1237)
  gamma_mult     : 1.8350 (init: 1.7611)
  sigma_mu       : 0.5516 (init: 0.5613)
  eta            : 0.9225 (init: 0.9033)
  eta_mult       : 0.8644 (init: 0.8877)
  phi            : 4.3193 (init: 4.4732)
  phi_mult       : 1.0879 (init: 1.0855)
  alpha          : 0.9293 (init: 0.9608)
  pi             : 0.6310 (init: 0.6144)
  lambda_        : 6.1532 (init: 5.7527)
  sigma_love     : 3.9296 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9284, data: 40.1000
  wage_level_w_35_44       : sim: 50.7922, data: 49.3000
  wage_level_m_25_34       : sim: 50.9942, data: 50.3000
  wage_level_m_35_44       : sim: 66.9932, data: 67.8000
  employment_rate_w_35_44  : sim: 63.5818, data: 64.0000
  employment_rate_m_35_44  : sim: 88.9270, data: 88.0000
  work_hours_w             : sim: 27.9032, data: 30.9548
  work_hours_m             : sim: 36.6765, data

In [9]:
model.save_par('calibrated_par')